# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object and print high-level info
metadata = dataset.metadata
print('Dataset name:', metadata.name)
print('Description:', metadata.description)
print('Version:', metadata.version)
print('Published date:', getattr(metadata, 'datePublished', None))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Entities are referenced **by their `@id`** for clarity and reproducibility.

In [ ]:
# List available record sets and their fields
print('Record Sets:')
for recset in metadata.record_sets:
    print(f"@id: {recset.id}")
    print(f"  Name: {recset.name}")
    print(f"  Description: {getattr(recset, 'description', '')}")
    print(f"  Fields:")
    for field in recset.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type if hasattr(field, 'data_type') else None}")
    print('---')

## 3. Data Extraction
Load data from specific record sets using their `@id`s. Store in DataFrames for convenient analysis.

In [ ]:
# Find all record set @ids
record_sets = [recset.id for recset in metadata.record_sets]
print('Record set @ids:', record_sets)

# Load all record sets into dataframes
dataframes = {}
for recset_id in record_sets:
    try:
        records = list(dataset.records(record_set=recset_id))
        dataframes[recset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set: {recset_id}")
    except Exception as e:
        print(f"Error loading {recset_id}: {e}")

# Print all columns for each loaded record set
for recset_id, df in dataframes.items():
    print(f'Columns for record set {recset_id}:', df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data cleaning and analysis steps. Use field `@id`s for filtering and transformations.

_Example below assumes there is a numeric field (e.g., log likelihood or coefficient) in one record set._

In [ ]:
# ---
# Please adjust these IDs according to outputs in previous cells if you wish to explore different fields

# Example for analysis: pick the first record set and its first numeric field
import numpy as np
primary_recset = None
numeric_field_id = None
group_field_id = None

# Try to auto-select a record set with a numeric field
for recset in metadata.record_sets:
    for field in recset.fields:
        dt = getattr(field, 'data_type', None)
        if dt in ['Integer', 'Float', 'Number']:
            primary_recset = recset.id
            numeric_field_id = field.id
            # Try to select a groupable field that is not the numeric field
            for gfield in recset.fields:
                if gfield.data_type in ['Text', 'String']:
                    group_field_id = gfield.id
                    break
            break
    if primary_recset: break

if (primary_recset is None) or (numeric_field_id is None):
    print("No suitable record set and numeric field found for EDA.")
else:
    df = dataframes[primary_recset]
    # Ensure numeric field is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Filter records (arbitrary threshold, e.g., > lower quartile)
    threshold = df[numeric_field_id].quantile(0.25)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by group_field_id
    if group_field_id and group_field_id in df:
        grouped_df = filtered_df.groupby(group_field_id).agg({numeric_field_id:'mean'})
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Here, we visualize the distribution of the selected numeric field and grouped means.

_You can further adjust field or grouping as needed depending on your analysis focus._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run if EDA cell above found suitable fields
if (primary_recset is not None) and (numeric_field_id is not None):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} in {primary_recset}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If grouping field exists, plot groupwise means
    if group_field_id and group_field_id in df:
        groupmeans = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        groupmeans = groupmeans.sort_values(ascending=False)
        plt.figure(figsize=(10,3))
        sns.barplot(x=groupmeans.index, y=groupmeans.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field/grouping for visualization.")

## 6. Conclusion
This notebook provided a guided exploration of the FAIR^2 dataset using `mlcroissant`, including:
- Listing all available record sets and their fields referenced by `@id`
- Loading data into Pandas DataFrames for each record set
- Conducting basic exploratory analysis on a selected numeric field
- Grouping and visualizing data using the `@id`s for reproducibility

You can now extend this notebook by repeating the analysis for other record sets or customizing your field selection for domain-specific exploration.